In [5]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [6]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [15]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [16]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model(
    save_name=None, tokenizer=TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer"), 
    device="cuda", depth=6, head_dim=128, context_size=4096, nb_heads_mult=1.0)
model.show_infos()

used auto save name: 'modelAuto_qxug-f8cw-786j-lhgl'
loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=128, window_pattern='SSSL')
1_589_356 total params (embeding: 327_680 | last layer: 81_920 | transformer: 1_179_744)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [17]:
dataset = svg_dataset.SVGDataset(
    OUR_DATASET_DIRECTORY, context_size=model.context_size,
    tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode)

In [18]:
model.train(
    dataset=dataset, batch_size=16, 
    nbEpoches=1, timeLimite=999_999, verbose="full")

starting epoch n°1
doing batches: 49.13 % done, rem: 1 min 3.3 sec 

KeyboardInterrupt: 